In [1]:
import sys
import os
from pathlib import Path
import pathlib
import pandas as pd
import numpy as np
import yaml

In [2]:
# Detecção automática de caminhos
notebook_path = pathlib.Path().absolute()
root_folder = notebook_path.parent

# Definir caminhos
SRC_DIR = root_folder / "src/data_processing"
CONFIG_DIR = root_folder / "config"
DATA_DIR = root_folder / "data"
COMPLETE_SERIES_DIR = DATA_DIR / "complete_series"
PROCESSED_DIR = DATA_DIR / "processed"
FORECAST_DIR = DATA_DIR / "forecast"
COMPLETE_SERIES_INFERENCE_DIR = DATA_DIR / "complete_series_inference"
PROCESSED_INFERENCE_DIR = DATA_DIR / "processed_inference"
FORECAST_INFERENCE_DIR = DATA_DIR / "forecast_inference"                   

In [3]:
# Detecção do root folder
notebook_path = pathlib.Path().absolute()
root_folder = notebook_path.parent

# Adicionar src ao path
SRC_DIR = root_folder
sys.path.append(str(SRC_DIR))

In [4]:
from src import (
    ConfigLoader, load_feature_config, load_split_config, load_config,
    HydroFeatureEngineer,   
    process_features,        
    load_observed_data,        
    load_forecast_data,    
    merge_observed_and_forecast, 
    InferenceConfig,          
    process_inference,          
    INTERNAL_COLUMN_NAMES,      
)

# Carregar configurações
print("\n⚙️ Carregando configurações...")
data_config = load_config(CONFIG_DIR / "data_config.yaml")


⚙️ Carregando configurações...


In [5]:
stations = data_config["stations"]
static_attributes_dict = data_config["static_attributes"]

# Extrair configurações de features
api_k_list = data_config['api_k_list']
precipitation_ma_windows = data_config['precipitation_ma']
precipitation_cumulative_windows = data_config['precipitation_cum']
forecast_ma_windows = data_config['forecast_ma']
forecast_cumulative_windows = data_config['forecast_cum']
evapotranspiration_ma_windows = data_config['evapotranspiration_ma']
anomaly_ma_windows = data_config['anomaly_ma']

# Flow e Climate window configs
flow_window_config = data_config["flow_window_config"]
climate_window_config = data_config["climate_window_config"]

# Temporal features
temporal_features = data_config["temporal_features"]

# Horizonte
horizon = data_config["horizon"]

# Split config
split_config = data_config["split_config"]

# Reference dates for inference
reference_dates = data_config["reference_dates"]

print("✅ Configurações de TREINO extraídas:")
print(f"   Estações: {stations}")
print(f"   API k: {api_k_list}")
print(f"   Precip MA: {precipitation_ma_windows}")
print(f"   Flow window config: {list(flow_window_config.keys())}")
print(f"   Horizonte: {horizon} dias")

✅ Configurações de TREINO extraídas:
   Estações: [10100000, 13150000, 14100000]
   API k: [0.7, 0.8, 0.85, 0.9, 0.92, 0.95]
   Precip MA: [3, 7, 15]
   Flow window config: [10100000, 13150000, 14100000]
   Horizonte: 15 dias


### Dataset treino

In [6]:
# Determinar o período dos dados automaticamente
print("\n🔍 Determinando período dos dados...")
try:
    # Carregar um arquivo de exemplo para determinar datas
    sample_file = COMPLETE_SERIES_DIR / f"{stations[0]}_complete_date.csv"
    sample_df = pd.read_csv(sample_file)
    sample_df['date'] = pd.to_datetime(sample_df['date'])
    
    start_date = sample_df['date'].min().strftime("%Y-%m-%d")
    end_date = sample_df['date'].max().strftime("%Y-%m-%d")
    
    print(f"  Período encontrado: {start_date} a {end_date}")
    print(f"  Total de dias: {(pd.to_datetime(end_date) - pd.to_datetime(start_date)).days}")
    
except Exception as e:
    print(f"⚠️  Não foi possível determinar o período automaticamente: {e}")
    # Valores padrão como fallback
    start_date = "1990-01-01"
    end_date = "2024-12-31"
    print(f"  Usando período padrão: {start_date} a {end_date}")

# Calcular datas de split automaticamente
loader = ConfigLoader()
split_dates = loader.calculate_split_dates(start_date, end_date)

print("\n📅 Datas de split calculadas automaticamente:")
print(f"  Train: {split_dates['train_start']} → {split_dates['train_end']} ({split_dates['train_days']} dias, {split_dates['train_days']/split_dates['total_days']*100:.1f}%)")
print(f"  Val:   {split_dates['train_end']} → {split_dates['val_end']} ({split_dates['val_days']} dias, {split_dates['val_days']/split_dates['total_days']*100:.1f}%)")
print(f"  Test:  {split_dates['val_end']} → {split_dates['test_end']} ({split_dates['test_days']} dias, {split_dates['test_days']/split_dates['total_days']*100:.1f}%)")

# Definir train_date_cutoff como o final do conjunto de treino
train_date_cutoff = split_dates['train_end']

print("\n📊 Configurações de janelas por tipo de feature:")
print(f"Precipitação (observada):")
print(f"  - Médias móveis: {precipitation_ma_windows}")
print(f"  - Acumulados: {precipitation_cumulative_windows}")
print(f"\nForecast de precipitação:")
print(f"  - Médias móveis: {forecast_ma_windows}")
print(f"  - Acumulados: {forecast_cumulative_windows}")
print(f"\nEvapotranspiração:")
print(f"  - Médias móveis: {evapotranspiration_ma_windows}")
print(f"\nAnomalias:")
print(f"  - Médias móveis: {anomaly_ma_windows}")
print(f"\nAPI:")
print(f"  - Valores k: {api_k_list}")


🔍 Determinando período dos dados...
  Período encontrado: 1995-01-01 a 2012-04-30
  Total de dias: 6329

📅 Datas de split calculadas automaticamente:
  Train: 1995-01-01 → 2011-06-18 (6012 dias, 95.0%)
  Val:   2011-06-18 → 2012-04-29 (316 dias, 5.0%)
  Test:  2012-04-29 → 2012-04-30 (1 dias, 0.0%)

📊 Configurações de janelas por tipo de feature:
Precipitação (observada):
  - Médias móveis: [3, 7, 15]
  - Acumulados: [3, 5, 7, 10]

Forecast de precipitação:
  - Médias móveis: [3, 7, 15]
  - Acumulados: [3, 5, 7, 10]

Evapotranspiração:
  - Médias móveis: [7, 14, 30]

Anomalias:
  - Médias móveis: [3, 7]

API:
  - Valores k: [0.7, 0.8, 0.85, 0.9, 0.92, 0.95]


In [7]:
# Mapeamento de colunas para os arquivos do set de TREINO
# (Arquivos que já possuem todas as colunas nomeadas corretamente)
train_column_mapping = {
    'flow': 'streamflow_m3s',
    'precip_obs': 'precipitation_chirps',
    'et_obs': 'potential_evapotransp_gleam',
    'precip_forecast': 'precipitation_forecast',
}

In [8]:
# Chamar função de processamento
print("\n🚀 Iniciando processamento de features...")
combined_df = process_features(
    input_dir=COMPLETE_SERIES_DIR,
    forecast_dir=FORECAST_DIR,
    output_dir=PROCESSED_DIR,
    station_ids=stations,
    api_k_list=api_k_list,
    precipitation_ma_windows=precipitation_ma_windows,
    precipitation_cumulative_windows=precipitation_cumulative_windows,
    forecast_ma_windows=forecast_ma_windows,
    forecast_cumulative_windows=forecast_cumulative_windows,
    evapotranspiration_ma_windows=evapotranspiration_ma_windows,
    anomaly_ma_windows=anomaly_ma_windows,
    train_date_cutoff=train_date_cutoff,
    output_filename="features_combined.csv",
)

print("\n✅ Processamento de features concluído!")
print(f"  Arquivo salvo: {PROCESSED_DIR / 'features_combined.csv'}")
print(f"  Train date cutoff usado: {train_date_cutoff}")


🚀 Iniciando processamento de features...
PROCESSAMENTO DE FEATURES
   Forçantes: P
📊 Carregando dados observados de 3 estações...
✓ 3 estações carregadas
🔮 Carregando dados de forecast...
✓ 3 estações com forecast carregadas
🔗 Fazendo merge observado + forecast...
   🗑️  Removidas 1 colunas de ET (forcings=P)
   🗑️  Removidas colunas desnecessárias: ['station_id', 'missing_data']
   🗑️  Removidas 1 colunas de ET (forcings=P)
   🗑️  Removidas colunas desnecessárias: ['station_id', 'missing_data']
   🗑️  Removidas 1 colunas de ET (forcings=P)
   🗑️  Removidas colunas desnecessárias: ['station_id', 'missing_data']
✓ 3 estações combinadas
⚙️  Criando features...
PROCESSANDO 3 ESTAÇÃO(ÕES)
   Forçantes: P
📊 Estação 10100000...
✅ OK - 36 features
📊 Estação 13150000...
✅ OK - 36 features
📊 Estação 14100000...
✅ OK - 36 features
✅ CONCLUÍDO - 112 colunas
✅ FEATURES CRIADAS COM SUCESSO
  Forçantes: P
  Estações processadas: 3
  Período: 1995-01-01 a 2012-04-30
  Total de dias: 6330
  Total de 

### Dataset Inferência

In [6]:
inference_column_mapping = {
    "date": "date",
    "flow": "mean_Q",  
    "precip": "mean_P",
    "et": None          
}

In [7]:
# Criar config de inferência
inference_config = InferenceConfig(
    observed_inference_dir=COMPLETE_SERIES_INFERENCE_DIR,
    forecast_inference_dir=FORECAST_INFERENCE_DIR,
    output_dir=PROCESSED_INFERENCE_DIR,
    station_ids=stations,
    reference_dates=reference_dates,
    column_mapping=inference_column_mapping,
    # Usar mesmas janelas do treino
    api_k_list=api_k_list,
    precipitation_ma_windows=precipitation_ma_windows,
    precipitation_cumulative_windows=precipitation_cumulative_windows,
    forecast_ma_windows=forecast_ma_windows,
    forecast_cumulative_windows=forecast_cumulative_windows,
    evapotranspiration_ma_windows=evapotranspiration_ma_windows,
    anomaly_ma_windows=anomaly_ma_windows,
    output_filename="features_combined_inference.csv"
)

# Processar
df_inference = process_inference(inference_config)

print(f"✅ Features de inferência criadas: {len(df_inference.columns)} colunas")

PROCESSAMENTO DE INFERÊNCIA HIDROLÓGICA
📅 Reference dates: ['2026-03-16']
🏭 Estações: [10100000, 13150000, 14100000]
🔧 Forçantes: P
📋 Mapeamento de colunas:
   date -> date
   flow -> mean_Q
   precip -> mean_P
   et: NÃO INCLUÍDO
📂 Carregando dados observados...
   ✅ Estação 10100000: 30 registros (até 2026-03-16)
   ✅ Estação 13150000: 30 registros (até 2026-03-16)
   ✅ Estação 14100000: 30 registros (até 2026-03-16)
📂 Carregando dados de forecast...
   ✅ Forecast 10100000: 16 registros (2026-03-17 até 2026-04-01)
   ✅ Forecast 13150000: 16 registros (2026-03-17 até 2026-04-01)
   ✅ Forecast 14100000: 16 registros (2026-03-17 até 2026-04-01)
🔗 Criando série contínua...
   📊 Estação 10100000:
      Observados: 2026-02-15 até 2026-03-16
      Reference date: 2026-03-16
      Forecast: 2026-03-17 até 2026-04-01
      ✅ Resultado: 30 dias observados (até 2026-03-16), 16 dias forecast (após 2026-03-16)
   📊 Estação 13150000:
      Observados: 2026-02-15 até 2026-03-16
      Reference date